# OCR -> HTML com OpenRouter

In [ ]:
MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"
IMAGE = "input/amostra_diversa/captura/300/01-1-cinza-300/p01.png"
KINDS = ("compact",)  # entradas do llm: compact, markdown, full_json
OUTPUT = "output/notebook"
CONTRACT = "contrato_campos.csv"  # contrato de campos: vai no prompt e valida os data-field do html
CPU_THREADS = 2
REASONING = False  # ligado, o qwen3.8 gasta 5–16 mil tokens pensando e pode truncar

In [ ]:
import os, sys
from pathlib import Path

try:  # import direto: "google.colab" in sys.modules pode dar falso mesmo no colab
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path("/content")
    if not (ROOT / IMAGE).is_file():  # no colab, envie só a imagem da nota
        from google.colab import files
        IMAGE = next(iter(files.upload()))
    if not (ROOT / CONTRACT).is_file():  # e o contrato de campos
        from google.colab import files
        CONTRACT = next(iter(files.upload()))
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    !pip install -q paddleocr==3.7.0 paddlepaddle==3.3.1 "paddlex[ocr]==3.7.2" openai python-dotenv beautifulsoup4 opencv-contrib-python
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")

from IPython.display import IFrame, display

## 0. Código da pipeline
Mesmo código do `pipeline.py`, que continua sendo usado pelo `main.py` local.

In [ ]:
# imports e caminhos
from contextlib import contextmanager
from datetime import datetime
from functools import cache
from html import escape
import json
import os
from pathlib import Path
import re
import threading
import time

from bs4 import BeautifulSoup
import psutil

BASE_DIR = ROOT  # definido na célula de preparo
OUTPUT_DIR = BASE_DIR / "output"
RUNS_DIR = BASE_DIR / "runs"
SUPPORTED_INPUTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".pdf"}
MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"
CONTRACT_PATH = BASE_DIR / "contrato_campos.csv"
FIELD_ATTR = "data-field"  # atributo que o parser lê em cada valor
REPEATING_GROUPS = {"item", "payment", "billing", "volumes"}  # várias ocorrências por nota
HTML_SUFFIX = {"compact": "_html.html", "markdown": "_from_md_html.html", "full_json": "_from_full_json_html.html"}

In [ ]:
# métricas

records, _active = [], []


@contextmanager
def stage(name):
    """mede tempo, cpu, pico de ram e tokens de llm de uma etapa."""
    record = {"stage": name, "peak_ram_mb": 0, "tokens_in": 0, "tokens_out": 0, "cost_usd": 0.0}
    stop, process = threading.Event(), psutil.Process()

    def sample():  # ru_maxrss marcou 7,9 GB onde a amostragem do rss viu 1,9 GB
        while True:
            record["peak_ram_mb"] = max(record["peak_ram_mb"], process.memory_info().rss / 2**20)
            if stop.wait(.05):
                return

    sampler = threading.Thread(target=sample, daemon=True)
    sampler.start()
    _active.append(record)
    cpu, started = time.process_time(), time.perf_counter()
    try:
        yield record
    finally:
        _active.pop()
        stop.set()
        sampler.join()
        record.update(seconds=time.perf_counter() - started, cpu_seconds=time.process_time() - cpu)
        records.append(record)


def record_llm(tokens_in, tokens_out, cost=0.0):
    if _active:
        _active[-1]["tokens_in"] += tokens_in or 0
        _active[-1]["tokens_out"] += tokens_out or 0
        _active[-1]["cost_usd"] += float(cost or 0)


def summary():
    import pandas as pd

    table = pd.DataFrame(records).groupby("stage", sort=False).agg(
        calls=("seconds", "size"), seconds=("seconds", "sum"), cpu_seconds=("cpu_seconds", "sum"),
        peak_ram_mb=("peak_ram_mb", "max"), tokens_in=("tokens_in", "sum"), tokens_out=("tokens_out", "sum"),
        cost_usd=("cost_usd", "sum"))
    table.insert(2, "cpu_%", 100 * table.pop("cpu_seconds") / table.seconds)
    return table


def report(label="pipeline"):
    """imprime o resumo, salva em runs/metrics_<timestamp>.json e zera as medições."""
    if not records:
        return None
    table = summary()
    print(table.round(4).to_string())
    RUNS_DIR.mkdir(exist_ok=True)
    path = RUNS_DIR / f"metrics_{datetime.now():%Y%m%d_%H%M%S_%f}.json"
    path.write_text(json.dumps({"label": label, "cpu_count": os.cpu_count(), "stages": records,
                                "summary": table.reset_index().to_dict("records")},
                               indent=2, default=lambda value: value.item()), encoding="utf-8")
    records.clear()
    print(f"Métricas: {path}")
    return path

In [ ]:
# ocr


def result_directories(results_dir):
    """separa intermediários e htmls finais em output/intermediarios e output/html."""
    root = Path(results_dir).resolve()
    if not root.is_relative_to(OUTPUT_DIR):
        return root / "intermediarios", root / "html"
    relative = root.relative_to(OUTPUT_DIR)
    if relative.parts and relative.parts[0] in {"html", "intermediarios"}:
        relative = Path(*relative.parts[1:])
    return OUTPUT_DIR / "intermediarios" / relative, OUTPUT_DIR / "html" / relative


def input_order(path, root):
    """prioriza imagens com maior resolução."""
    first = path.relative_to(root).parts[0]
    return -int(first) if first in {"100", "150", "200", "300"} else 0, str(path.relative_to(root))


def select_inputs(root, dpi=None, limit=0):
    files = sorted((path for path in root.rglob("*") if path.is_file() and path.suffix.lower() in SUPPORTED_INPUTS
                    and (not dpi or path.relative_to(root).parts[0] == str(dpi))), key=lambda p: input_order(p, root))
    return files[:limit] if limit else files


# textline_orientation=False: ligado, gira por engano linhas das notas em paisagem e o ocr lê lixo
# (campos 66,7% → 88,9% na nota 02-1-cinza-300, mesmo tempo)
def load_engine(lang="pt", doc_orientation=True, textline_orientation=False, det_max_side=1280, cpu_threads=2):
    from paddleocr import PPStructureV3

    return PPStructureV3(
        lang=None if lang == "default" else lang, ocr_version=None if lang == "default" else "PP-OCRv5",
        use_doc_orientation_classify=doc_orientation, use_textline_orientation=textline_orientation,
        use_doc_unwarping=False,  # capturas sintéticas já são planas
        enable_mkldnn=False,  # onednn falha no paddle 3.3.1 (ConvertPirAttribute2RuntimeAttribute)
        cpu_threads=cpu_threads,  # 2 foi o mais rápido no i7-1355U
        use_formula_recognition=False, use_chart_recognition=False, use_seal_recognition=False,
        use_table_recognition=True, text_det_limit_side_len=det_max_side, text_det_limit_type="max")


def simplify_result(res):
    """json compacto: só tipo, posição e conteúdo de cada bloco."""
    data = json.loads(res.json) if isinstance(res.json, str) else res.json
    return [{"id": i, "type": block.get("block_label"), "bbox": block.get("block_bbox"),
             "content": block.get("block_content")}
            for i, block in enumerate(data.get("res", data).get("parsing_res_list", []))]


def prepare_image(source, destination, min_width=0):
    """amplia até min_width; imagem já larga volta como está."""
    from PIL import Image

    with Image.open(source) as image:
        if image.width >= min_width:
            return source
        destination.parent.mkdir(parents=True, exist_ok=True)
        image.resize((min_width, round(image.height * min_width / image.width)),
                     Image.Resampling.LANCZOS).convert("RGB").save(destination)
        return destination


def process_file(engine, input_path, input_root, output, min_width=1600, ruled_tables=True):
    """roda o ocr em uma imagem ou pdf, salva as saídas e retorna (pasta, status)."""
    relative = input_path.relative_to(input_root)
    folder = result_directories(output)[0] / relative.parent / f"{input_path.stem}_{input_path.suffix[1:].lower()}"
    folder.mkdir(parents=True, exist_ok=True)
    status = "ok"
    try:
        source = input_path
        if input_path.suffix.lower() != ".pdf":
            with stage("ocr:preparar_imagem"):
                source = prepare_image(input_path, folder / "_inputs" / f"{input_path.stem}.png", min_width)
        with stage("ocr:predict"):
            pages = list(engine.predict(str(source)))
        if not pages:
            raise ValueError("OCR não retornou páginas")
        for page, res in enumerate(pages, start=1):
            if ruled_tables:
                with stage("ocr:reparar_tabelas"):
                    repair_tables(res, source)
            with stage("ocr:salvar"):
                res.save_to_json(save_path=str(folder))
                res.save_to_markdown(save_path=str(folder))
                res.save_to_html(save_path=str(folder))
                (folder / f"{input_path.stem}_page_{page:03d}_layout.json").write_text(
                    json.dumps(simplify_result(res), ensure_ascii=False, indent=2), encoding="utf-8")
    except Exception as exc:
        status = "failed"
        print(f"Falha em {input_path.name}: {exc}", flush=True)
    (folder / "ocr_run.json").write_text(json.dumps({"status": status}), encoding="utf-8")
    return folder, status

In [ ]:
# tabelas pelas bordas visíveis


def _boundaries(values, tolerance):
    """agrupa coordenadas próximas e devolve a média de cada grupo."""
    groups = []
    for value in sorted(values):
        if groups and value - groups[-1][-1] <= tolerance:
            groups[-1].append(value)
        else:
            groups.append([value])
    return [round(sum(group) / len(group)) for group in groups]


def _overlap(box, cell):
    x1, y1, x2, y2 = box
    a, b, c, d = cell
    return max(0, min(x2, c) - max(x1, a)) * max(0, min(y2, d) - max(y1, b)) / max(1, (x2-x1)*(y2-y1))


def detect_cells(image):
    import cv2
    import numpy as np

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    _, ink = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    height, width = gray.shape
    horizontal = cv2.morphologyEx(ink, cv2.MORPH_OPEN, np.ones((1, max(20, width // 40)), np.uint8))
    vertical = cv2.morphologyEx(ink, cv2.MORPH_OPEN, np.ones((max(20, height // 60), 1), np.uint8))
    borders = cv2.morphologyEx(horizontal | vertical, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    _, _, stats, _ = cv2.connectedComponentsWithStats(255 - borders)
    return [[int(x), int(y), int(x+w), int(y+h)] for x, y, w, h, area in stats[1:]
            if w >= 12 and h >= 8 and area / (w*h) > .8 and x > 0 and y > 0 and x+w < width and y+h < height]


def ruled_table(image, ocr, bbox, cells=None):
    """reconstrói a tabela pela grade visível; sem bordas confiáveis, retorna None."""
    cells = [cell for cell in (detect_cells(image) if cells is None else cells) if _overlap(cell, bbox) > .9]
    if len(cells) < 8:
        return None
    eligible = [(i, text, box) for i, (text, box) in enumerate(zip(ocr.get("rec_texts", []), ocr.get("rec_boxes", [])))
                if text.strip() and _overlap(box, bbox) > .5]
    if not eligible:
        return None
    assigned, loose = [[] for _ in cells], []
    for i, text, box in eligible:
        hits = sorted([(_overlap(box, cell), j) for j, cell in enumerate(cells)], reverse=True)
        if hits[0][0] >= .55 and (len(hits) == 1 or hits[1][0] < .25):
            assigned[hits[0][1]].append((box[1], box[0], i, text))
        else:
            loose.append((box[1], i, text))
    matched = len(eligible) - len(loose)
    if matched / len(eligible) < .65:
        return None

    tolerance = max(3, round(image.shape[1] / 700))
    xs = _boundaries([v for c in cells for v in (c[0], c[2])], tolerance)
    ys = _boundaries([v for c in cells for v in (c[1], c[3])], tolerance)
    nearest = lambda value, edges: min(range(len(edges)), key=lambda i: abs(edges[i] - value))
    grid = [(nearest(c[0], xs), nearest(c[1], ys), nearest(c[2], xs), nearest(c[3], ys)) for c in cells]
    if any(left >= right or top >= bottom for left, top, right, bottom in grid):
        return None
    # faixas horizontais que nenhuma célula atravessa; faixas vizinhas com as mesmas colunas viram uma tabela
    cuts = [y for y in range(len(ys)) if not any(top < y < bottom for _, top, _, bottom in grid)]
    bands = []
    for lo, hi in zip(cuts, cuts[1:]):
        members = [i for i, (_, top, _, bottom) in enumerate(grid) if lo <= top < bottom <= hi]
        if not members:
            continue
        signature = tuple(sorted({v for i in members for v in (grid[i][0], grid[i][2])}))
        if bands and bands[-1]["end"] == lo and bands[-1]["signature"] == signature:
            bands[-1].update(end=hi, members=bands[-1]["members"] + members)
        else:
            bands.append({"start": lo, "end": hi, "signature": signature, "members": members})

    fragments = []
    for band in bands:
        members = band["members"]
        if not any(assigned[i] for i in members):
            continue
        columns = list(band["signature"])
        rows = sorted({v for i in members for v in (grid[i][1], grid[i][3])})
        html = ["<table><tbody>"]
        for top in rows[:-1]:
            html.append("<tr>")
            occupied = {x for i in members if grid[i][1] < top < grid[i][3]
                        for x in range(columns.index(grid[i][0]), columns.index(grid[i][2]))}
            starts = {columns.index(grid[i][0]): i for i in members if grid[i][1] == top}
            x = 0
            while x < len(columns)-1:
                if x in occupied or x not in starts:
                    html.append("" if x in occupied else "<td></td>")
                    x += 1
                    continue
                i = starts[x]
                left, _, right, bottom = grid[i]
                colspan, rowspan = columns.index(right) - columns.index(left), rows.index(bottom) - rows.index(top)
                attrs = (f' colspan="{colspan}"' if colspan > 1 else "") + (f' rowspan="{rowspan}"' if rowspan > 1 else "")
                html.append(f"<td{attrs}>{'<br>'.join(escape(line[3]) for line in sorted(assigned[i]))}</td>")
                x += colspan
            html.append("</tr>")
        html.append("</tbody></table>")
        fragments.append((ys[band["start"]], "".join(html)))
    fragments.extend((y, f"<p>{escape(text)}</p>") for y, _, text in loose)  # textos sem posição segura
    return {"html": "\n".join(fragment for _, fragment in sorted(fragments, key=lambda part: part[0])),
            "cells": cells, "matched_lines": matched, "loose_line_ids": [i for _, i, _ in loose]}


def repair_tables(res, input_path):
    blocks = [block for block in res["parsing_res_list"] if block.label == "table"]
    if not blocks:
        return
    import cv2

    preprocessed = res.get("doc_preprocessor_res", {})
    image = preprocessed.get("output_img")
    if image is None and not preprocessed.get("angle", 0):
        image = cv2.imread(str(input_path))
    if image is None:
        return
    cells = detect_cells(image)
    for block, table in zip(blocks, res["table_res_list"]):
        if repaired := ruled_table(image, res["overall_ocr_res"], block.bbox, cells):
            block.content = table["pred_html"] = repaired["html"]
            table["cell_box_list"] = repaired["cells"]

In [ ]:
# html

BASE_PROMPT = """Return only a complete HTML document with closed tags, no Markdown.
Preserve all recognized text exactly and in order; never correct, omit or invent content.
Keep labels with values and the relative layout of fields and sections.
Preserve every table, row, td/th, rowspan and colspan; do not merge unrelated cells.
"""

FIELD_RULES = """Field tagging: every value that matches a field of the CONTRACT below must carry data-field="<campo>" with the exact campo name.
- Put the attribute on the smallest element that holds only the value (td, th, p or span), never on a label, header or section.
- If a cell holds label and value together, wrap only the value in <span data-field="...">; the label stays outside and all text and order stay unchanged.
- One element, one field. If one text holds several fields (page "1/2", the issuer block, an installment line), wrap each part in its own span. Never nest tagged elements.
- Tag every CONTRACT field whose value is printed. Use only campo names from the CONTRACT; never invent names or add text just to hold a tag.
- Decide by the section where the value is printed: the same label can be an item (item.*, one row per product), a totals field (totals.*), or belong to issuer, recipient or carrier.
- item.*, payment.*, billing.* and volumes.* repeat: tag every row. If product data arrives as loose <p> paragraphs instead of table rows, do not tag item.* at all: a wrong tag is worse than none.
- Never delete, shorten or reorder any text, label or cell of the input; only add data-field attributes and spans.
- Respect "only if": CNPJ/CPF of recipient and carrier is one box (11 digits is *.cpf, 14 characters is *.cnpj, empty is *.document); item CST vs CSOSN follows the column header; exit vs entry date/time follows invoice.operation_type (1 saída, 0 entrada), the digit between "0 - ENTRADA" and "1 - SAÍDA".
- Fields marked (NFC-e) exist only on NFC-e receipts; DANFE uses invoice.issue_date.
"""

HTML_STYLE = """<style>
@page { size: A4 landscape; margin: 10mm; }
* { box-sizing: border-box; }
html { background: #eef1f5; }
body { margin: 24px auto; max-width: 1440px; padding: 24px;
       background: #fff; color: #202936; font: 14px/1.5 Arial, sans-serif;
       box-shadow: 0 4px 24px #18263914; }
.invoice-block + .invoice-block { margin-top: 20px; }
.table-scroll { overflow-x: auto; max-width: 100%; }
table { width: 100%; border-collapse: collapse; table-layout: auto; text-align: left; }
td, th { border: 1px solid #c4ccd5; padding: 8px 10px; vertical-align: top;
         overflow-wrap: break-word; min-width: 48px; }
th { background: #e8edf3; font-weight: 700; }
tr:nth-child(even) > td { background: #f7f9fc; }
td.numeric, th.numeric { white-space: nowrap; text-align: right; font-variant-numeric: tabular-nums; }
tr { break-inside: avoid; }
p { margin: 0 0 12px; white-space: pre-wrap; }
img { max-width: 100%; height: auto; }
@media (max-width: 600px) { body { margin: 0; padding: 12px; } }
@media print {
  html, body { background: #fff; }
  body { margin: 0; padding: 0; max-width: none; font-size: 8pt; box-shadow: none; }
  .table-scroll { overflow: visible; }
  td, th { padding: 3px; min-width: 0; }
  td.numeric, th.numeric { white-space: normal; }
}
</style>"""


def source_path(layout_path, kind):
    stem = layout_path.name.rsplit("_page_", 1)[0]
    return {"compact": layout_path, "markdown": layout_path.with_name(f"{stem}.md"),
            "full_json": layout_path.with_name(f"{stem}_res.json")}[kind]


def output_path(layout_path, kind, results_dir):
    intermediates_dir, html_dir = result_directories(results_dir)
    name = layout_path.name.removesuffix("_layout.json") + HTML_SUFFIX[kind]
    return (html_dir / layout_path.relative_to(intermediates_dir)).with_name(name)


def prepare_content(kind, path):
    content = path.read_text(encoding="utf-8")
    if kind == "markdown":
        return content
    data = json.loads(content)
    if kind == "full_json":
        data = [{"type": block.get("block_label"), "bbox": block.get("block_bbox"), "content": block.get("block_content")}
                for block in data.get("res", data).get("parsing_res_list", [])]
    return json.dumps(data, ensure_ascii=False, separators=(",", ":"))


def clean_fragment(content, keep_classes=False):
    """remove estilos e documentos aninhados; preserva células e mesclagens."""
    soup = BeautifulSoup(content, "html.parser")
    for tag in soup.find_all(["head", "style", "script", "meta", "link"]):
        tag.decompose()
    for tag in soup.find_all(["html", "body"]):
        tag.unwrap()
    allowed = {"rowspan", "colspan", FIELD_ATTR} | ({"class"} if keep_classes else set())
    for tag in soup.find_all(True):
        tag.attrs = {key: value for key, value in tag.attrs.items() if key in allowed}
    return soup


def render_source(kind, content):
    """html direto do ocr, sem llm."""
    fragments = [content] if kind == "markdown" else [
        str(block.get("content") or "") if block.get("type") == "table"
        else f"<p>{escape(str(block.get('content') or ''))}</p>" for block in json.loads(content)]
    soup = clean_fragment("\n".join(f"<section>{part}</section>" for part in fragments))
    for section in soup.find_all("section", recursive=False):
        section["class"] = "invoice-block"
    return add_presentation(soup)


def parse_html(html):
    """resposta do llm -> fragmento limpo (sem cercas de markdown, estilos e documento aninhado)."""
    return clean_fragment(re.sub(r"^```(?:html)?\s*|\s*```$", "", html.strip(), flags=re.IGNORECASE), keep_classes=True)


def add_presentation(html):
    """aplica o estilo sem alterar o conteúdo das células."""
    soup = html if not isinstance(html, str) else parse_html(html)
    for cell in soup.find_all(["td", "th"]):
        if re.fullmatch(r"[+−-]?[\d.,%/ -]+", cell.get_text(strip=True)):
            cell["class"] = ["numeric"]
    for table in soup.find_all("table"):
        table.wrap(soup.new_tag("div", attrs={"class": "table-scroll", "tabindex": "0", "role": "region",
                                              "aria-label": "Tabela da nota fiscal"}))
    return ('<!doctype html><html lang="pt-BR"><head><meta charset="utf-8">'
            '<meta name="viewport" content="width=device-width, initial-scale=1">'
            f'<title>Nota fiscal</title>{HTML_STYLE}</head><body>{soup}</body></html>')


@cache
def _read_contract(path):
    import csv

    with open(path, encoding="utf-8-sig", newline="") as file:
        rows = [{key: (value or "").strip() for key, value in row.items()} for row in csv.DictReader(file)]
    if not rows or not {"campo", "descricao", "xml", "condicao"} <= rows[0].keys():
        raise ValueError(f"Contrato inválido: {path} (colunas esperadas: campo, descricao, xml, condicao)")
    return tuple(rows)


def load_contract(path=None):
    """contrato de campos (campo, descricao, xml, condicao) lido do csv."""
    return _read_contract(Path(path or CONTRACT_PATH))


def field_instructions(contract, with_xml=False):
    """regras de data-field + o contrato, em texto para o prompt (o caminho xml é opcional: alonga o prompt)."""
    lines = [f"{row['campo']} — {row['descricao']}" + (f" [xml: {row['xml']}]" if with_xml and row["xml"] else "")
             + (f" (only if: {row['condicao']})" if row["condicao"] else "") for row in contract]
    return f"{FIELD_RULES}\nCONTRACT (campo — description):\n" + "\n".join(lines) + "\n\n"


FIELD_FORMATS = {  # formato impossível para o campo = marcação errada; o valor é lido sem espaços nem <br>
    "item.origin": r"[0-8]", "item.cfop": r"\d{4}", "item.ncm": r"\d{8}", "item.cst": r"\d{2,3}",
    "item.csosn": r"\d{2,4}", "item.unit": r"[A-Za-z][A-Za-z0-9]{0,5}",
    **dict.fromkeys(("item.quantity", "item.unit_price", "item.total_price", "item.discount", "item.icms_base",
                     "item.icms", "item.ipi", "item.icms_rate", "item.ipi_rate", "item.approximate_tax"),
                    r"[\d.]*\d,\d+")}


def lost_words(kind, content, soup, limit=20):
    """palavras do ocr que não estão no html do llm: o modelo não deveria apagar texto."""
    from collections import Counter

    text = content if kind == "markdown" else " ".join(  # aside_text é ruído de margem que o modelo pode descartar
        str(b.get("content") or "") for b in json.loads(content) if b.get("type") != "aside_text")
    expected = Counter(BeautifulSoup(text, "html.parser").get_text(" ").split())
    found = Counter(soup.get_text(" ").split())
    return dict((expected - found).most_common(limit))


def validate_fields(soup, contract):
    """confere os data-field do llm contra o contrato e corrige o que o próprio texto decide.

    Remove nomes fora do contrato e marcações aninhadas; reclassifica CPF/CNPJ/documento pelo valor impresso
    e entrada/saída pelo tipo de operação. Retorna um relatório (o html é alterado no lugar)."""
    from collections import Counter

    names = {row["campo"] for row in contract}
    report = {"invalid": [], "nested": [], "renamed": [], "format": [], "repeated_scalar": [], "counts": {}}
    text = lambda tag: tag.get_text(" ", strip=True)

    def rename(tag, new):
        if new != tag[FIELD_ATTR]:
            report["renamed"].append([tag[FIELD_ATTR], new])
            tag[FIELD_ATTR] = new

    for tag in soup.find_all(attrs={FIELD_ATTR: True}):
        name = " ".join(str(tag[FIELD_ATTR]).split())
        if name in names:
            tag[FIELD_ATTR] = name
        else:
            report["invalid"].append(name)
            del tag[FIELD_ATTR]
    for tag in soup.find_all(attrs={FIELD_ATTR: True}):  # fica o mais interno, que é o mais específico
        if tag.find(attrs={FIELD_ATTR: True}):
            report["nested"].append(tag[FIELD_ATTR])
            del tag[FIELD_ATTR]

    for group in ("recipient", "carrier"):  # a nota imprime uma caixa só: 11 dígitos CPF, 14 caracteres CNPJ
        for tag in soup.find_all(attrs={FIELD_ATTR: re.compile(rf"^{group}\.(cpf|cnpj|document)$")}):
            value = re.sub(r"[^0-9A-Za-z]", "", text(tag))  # cnpj alfanumérico (2026) também tem 14
            if not value:
                rename(tag, f"{group}.document")
            elif len(value) == 14:
                rename(tag, f"{group}.cnpj")
            elif len(value) == 11 and value.isdigit():
                rename(tag, f"{group}.cpf")

    operation = soup.find(attrs={FIELD_ATTR: "invoice.operation_type"})
    if operation:
        digits = set(re.findall(r"(?<!\d)[01](?!\d)", text(operation)))
        if len(digits) == 1:  # se o texto traz "0 entrada 1 saída" juntos, é só o rótulo: não decide
            kind = "exit" if digits == {"1"} else "entry"
            for tag in soup.find_all(attrs={FIELD_ATTR: re.compile(r"^invoice\.(exit|entry)_(date|time)$")}):
                rename(tag, f"invoice.{kind}_{tag[FIELD_ATTR].rsplit('_', 1)[1]}")

    for tag in soup.find_all(attrs={FIELD_ATTR: tuple(FIELD_FORMATS)}):  # valor que não cabe no campo: tira a marcação
        value = re.sub(r"\s+", "", text(tag))
        if value and not re.fullmatch(FIELD_FORMATS[tag[FIELD_ATTR]], value):
            report["format"].append([tag[FIELD_ATTR], text(tag)])
            del tag[FIELD_ATTR]

    counts = Counter(tag[FIELD_ATTR] for tag in soup.find_all(attrs={FIELD_ATTR: True}))
    report["counts"] = dict(counts)
    report["repeated_scalar"] = sorted(name for name, n in counts.items()
                                       if n > 1 and name.split(".")[0] not in REPEATING_GROUPS)
    return report


def list_fields(html_path):
    """(campo, texto) de cada elemento marcado, na ordem do documento: o que o parser vai ler."""
    soup = BeautifulSoup(Path(html_path).read_text(encoding="utf-8"), "html.parser")
    return [(tag[FIELD_ATTR], tag.get_text(" ", strip=True)) for tag in soup.find_all(attrs={FIELD_ATTR: True})]


@cache
def _client():
    from dotenv import load_dotenv
    from openai import OpenAI

    load_dotenv(BASE_DIR / ".env")
    return OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"], max_retries=5)


REMINDER = ("\n\nReminder: apply the data-field rules and the CONTRACT above. "
            "An answer without data-field attributes is invalid.")


CALLS = []  # prompt e resposta crua de cada chamada ao llm


def ask_llm(kind, content, model=MODEL, num_predict=16384, reasoning=False, contract=None):
    source = ("Input: PaddleOCR Markdown with HTML tables." if kind == "markdown" else
              "Input: OCR JSON blocks. Use content as text/table markup, type as label, bbox for layout only; hide coordinates.")
    prompt = (f"{BASE_PROMPT}{field_instructions(contract) if contract else ''}"
              f"{source}\n\n{content}{REMINDER if contract else ''}")
    response = _client().chat.completions.create(
        model=model, temperature=0, max_tokens=num_predict,
        messages=[{"role": "user", "content": prompt}],
        extra_body={"reasoning": {"enabled": reasoning}})
    record_llm(response.usage.prompt_tokens, response.usage.completion_tokens, getattr(response.usage, "cost", 0))
    choice = response.choices[0]
    CALLS.append({"model": model, "prompt": prompt, "raw": choice.message.content,
                  "finish_reason": choice.finish_reason})  # antes dos raise, para guardar também respostas truncadas
    if choice.finish_reason == "length":
        raise ValueError("HTML truncado: aumente num-predict.")
    if not choice.message.content:
        raise ValueError("O modelo não retornou conteúdo.")
    return choice.message.content


def write_html(layout_path, kind, results_dir, engine="direct", model=None, num_predict=16384, reasoning=False,
               datafields=True):
    """gera um html final (engine direct ou openrouter); o log fica nos intermediários.

    datafields: o llm marca cada valor com data-field conforme o contrato, que depois é validado.
    O engine direct não usa llm e portanto não marca campos."""
    source = source_path(layout_path, kind)
    if not source.is_file():
        print(f"Entrada {kind} ausente: {source}")
        return None
    destination = output_path(layout_path, kind, results_dir)
    destination.parent.mkdir(parents=True, exist_ok=True)
    log = {"source": str(source), "engine": engine, "status": "failed"}
    with stage(f"html:{engine}:{kind}") as record:
        try:
            content = prepare_content(kind, source)
            if engine == "direct":
                html = render_source(kind, content)
            else:
                contract = load_contract() if datafields else None
                for attempt in (1, 2):  # modelo pequeno às vezes ignora o data-field: uma nova tentativa
                    soup = parse_html(ask_llm(kind, content, model or MODEL, num_predict, reasoning, contract))
                    if not contract:
                        break
                    log["datafields"] = report = validate_fields(soup, contract)
                    print(f"Datafields: {len(report['counts'])} campos, {sum(report['counts'].values())} marcações; "
                          f"{len(report['invalid'])} inválidas, {len(report['nested'])} aninhadas, "
                          f"{len(report['renamed'])} reclassificadas, {len(report['format'])} fora de formato, "
                          f"{len(report['repeated_scalar'])} repetidas",
                          flush=True)
                    log["lost_words"] = lost = lost_words(kind, content, soup)
                    if lost:
                        print(f"Atenção: o modelo apagou texto do OCR: {lost}", flush=True)
                    if report["counts"]:
                        break
                    print("Nenhum data-field na resposta" + ("; tentando de novo." if attempt == 1 else "."), flush=True)
                else:
                    raise ValueError("O modelo não marcou nenhum data-field. Teste outro modelo.")
                html = add_presentation(soup)
            destination.write_text(html, encoding="utf-8")
            if engine != "direct":
                log.update(model=model or MODEL, **{k: record[k] for k in ("tokens_in", "tokens_out", "cost_usd")})
            log["status"] = "accepted"
            print(f"HTML ({kind}): {destination}", flush=True)
        except Exception as exc:
            log["error"] = str(exc)
            print(f"Falha no fluxo {kind} para {source.name}: {exc}", flush=True)
    log["duration_seconds"] = round(record["seconds"], 3)
    layout_path.with_name(destination.name).with_suffix(".generation.json").write_text(
        json.dumps(log, ensure_ascii=False, indent=2), encoding="utf-8")
    return log

In [ ]:
image, output = ROOT / IMAGE, ROOT / OUTPUT
CONTRACT_PATH = ROOT / CONTRACT
for path in (image, CONTRACT_PATH):  # falha aqui, com a causa clara, e não no meio do ocr
    if not path.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}. Ajuste IMAGE/CONTRACT na primeira célula ou envie o arquivo para {ROOT}.")
show = lambda prefix: display(summary().filter(like=prefix, axis=0).round(2))

## 1. OCR

In [ ]:
with stage("ocr:carregar_modelo"):
    engine = load_engine(cpu_threads=CPU_THREADS)
folder, status = process_file(engine, image, image.parent, output)
if status != "ok":
    raise RuntimeError(f"OCR falhou em {image.name}; veja a mensagem acima.")
layout = next(folder.glob("*_layout.json"))
print(status, layout)
show("ocr")

## 2. HTML via OpenRouter

In [ ]:
CALLS.clear()

logs = [write_html(layout, kind, output, engine="openrouter", model=MODEL, reasoning=REASONING)
        for kind in KINDS]
display(logs)
for log in logs:  # write_html não levanta erro: falha vira log com status "failed"
    if not log or log["status"] != "accepted":
        raise RuntimeError(f"HTML não gerado: {log and log.get('error')}")
show("html")

In [ ]:
IFrame(os.path.relpath(output_path(layout, KINDS[0], output)), width="100%", height=600)

In [ ]:
import shutil

EXPORT = ROOT / "exemplos" / f"{image.parent.name}_{image.stem}"
EXPORT.mkdir(parents=True, exist_ok=True)

shutil.copy(image, EXPORT / image.name)
for n, call in enumerate(CALLS, 1):  # mais de uma se houve retry
    (EXPORT / f"tentativa{n}_prompt.txt").write_text(call["prompt"], encoding="utf-8")
    (EXPORT / f"tentativa{n}_resposta_crua.html").write_text(call["raw"] or "", encoding="utf-8")

final = output_path(layout, KINDS[0], output)
generation = layout.with_name(final.name).with_suffix(".generation.json")
for source, name in ((final, "html_final.html"), (generation, "log_validacao.json")):
    if source.is_file():  # em caso de falha o html final não existe, mas o resto ainda sai
        shutil.copy(source, EXPORT / name)

if IN_COLAB:
    !cd {ROOT} && zip -qr /content/exemplos.zip exemplos
    from google.colab import files
    files.download("/content/exemplos.zip")

### Conferência dos datafields
Cada valor marcado no html, na ordem do documento (o que o parser vai ler).

In [ ]:
import pandas as pd

fields = pd.DataFrame(list_fields(output_path(layout, KINDS[0], output)), columns=["campo", "valor"])
display(fields)
print("Campos do contrato sem marcação:", sorted(set(r["campo"] for r in load_contract()) - set(fields.campo)))

## 3. Resumo

In [ ]:
report("notebook");

In [ ]:
if IN_COLAB:  # baixa output/ e runs/ num zip
    !cd {ROOT} && zip -qr /content/colab_resultados.zip output runs
    from google.colab import files
    files.download("/content/colab_resultados.zip")